# Tutorial 05 — Group-Level Statistics

After running MVPA on individual subjects, we aggregate to the group level:

1. **Decoding accuracy** — group mean ± SD and a one-sample t-test against chance
2. **Importance maps** — voxel-wise one-sample t-test + FDR correction across subjects

**Inputs** (one file per subject, produced by Tutorial 04):
- `sub-XX_mvpa_results.npz` — saved pipeline results including `mean_acc`
- `sub-XX_importance_map.nii.gz` — voxel-wise SVM weight map in MNI space

**Outputs:** group accuracy summary, thresholded group t-map, glass brain figure

---
## Section 0 — Setup

In [ ]:
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from statsmodels.stats.multitest import multipletests
from nilearn import image, plotting
from nilearn.masking import compute_brain_mask

print('Imports OK')

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
RESULTS_DIR  = Path('/pl/active/courses/2026_summer/neuroclass2026/ds000105/derivatives/mvpa/all_conditions_lss/')
SUBJECTS     = [f'sub-{i}' for i in range(1, 7)]   # update to your subject list
CHANCE_LEVEL = 1/8    # binary classification: 50%; 6-way: 1/6 ≈ 0.167
ALPHA_FDR    = 0.05    # FDR threshold
OUT_DIR      = Path('group_results')
OUT_DIR.mkdir(exist_ok=True)

print(f'Subjects ({len(SUBJECTS)}): {SUBJECTS}')

---
## Section 1 — Group Decoding Accuracy

Load each subject's mean leave-one-run-out accuracy and test whether the group
mean is above chance with a one-sample t-test.

In [ ]:
def load_accuracies(results_dir, subjects):
    # Load per-subject accuracy from saved .npz result files
    accs = {}
    for sub in subjects:
        path = results_dir / sub / f'accuracy_metrics.csv'
        if not path.exists():
            print(f'  WARNING: {path} not found — skipping')
            continue
        data = pd.read_csv(path)
        accs[sub] = float(data['mean_accuracy'].values[0])
        print(f'  {sub}: {accs[sub]:.3f}')
    return accs


acc_dict = load_accuracies(RESULTS_DIR, SUBJECTS)
acc_vals = np.array(list(acc_dict.values()))

# Group stats
group_mean = acc_vals.mean()
group_sd   = acc_vals.std(ddof=1)
group_sem  = group_sd / np.sqrt(len(acc_vals))

t_stat, p_val = stats.ttest_1samp(acc_vals, popmean=CHANCE_LEVEL)

print(f'\nGroup accuracy: {group_mean:.3f} ± {group_sd:.3f} (SD)')
print(f'One-sample t-test vs chance ({CHANCE_LEVEL}): '
      f't({len(acc_vals)-1}) = {t_stat:.2f},  p = {p_val:.4f}')
print('Warning! t-test should not be used for group accuracy statisitcs given the clipped range of Accuracy [0-1]')

In [ ]:
# ── Accuracy bar chart ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))

x = np.arange(len(acc_dict))
ax.bar(x, list(acc_dict.values()), color='#2B9B8E', edgecolor='white',
       linewidth=0.8, alpha=0.85, label='Subject accuracy')
ax.axhline(CHANCE_LEVEL, color='#C0392B', lw=1.8, ls='--', label='Chance')
ax.axhline(group_mean,   color='#2C3E50', lw=2.0, ls='-',
           label=f'Group mean = {group_mean:.3f}')
ax.fill_between([-0.5, len(x) - 0.5],
                group_mean - group_sem, group_mean + group_sem,
                color='#2C3E50', alpha=0.15, label='±SEM')

ax.set_xticks(x)
ax.set_xticklabels(list(acc_dict.keys()), rotation=45, ha='right', fontsize=9)
ax.set_ylabel('LORO-CV Accuracy', fontsize=11)
ax.set_ylim(0, 1.05)
ax.set_title(f'Group Decoding Accuracy  '
             f't({len(acc_vals)-1})={t_stat:.2f}, p={p_val:.3f}',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
fig.savefig(OUT_DIR / 'group_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 2 — Group Importance Map

Each subject's importance map is a 3-D volume of mean |SVM weights| in MNI space.
We stack them into a **subjects × voxels** matrix, run a voxel-wise one-sample
t-test against zero, and apply FDR correction.

In [ ]:
def load_importance_maps(results_dir, subjects):
    imgs, loaded = [], []
    for sub in subjects:
        path = results_dir / sub / f'{sub}_importance_map_MNIspace.nii.gz'
        if not path.exists():
            print(f'  WARNING: {path} not found — skipping')
            continue
        imgs.append(nib.load(path))
        loaded.append(sub)
        print(f'  {sub}: {imgs[-1].shape}')
    print(f'Loaded {len(imgs)} importance maps')
    return imgs, loaded


imp_imgs, imp_subs = load_importance_maps(RESULTS_DIR, SUBJECTS)
ref_img = imp_imgs[0]

In [ ]:
# ── Brain mask ────────────────────────────────────────────────────────────────
# Use a whole-brain mask computed from the mean importance map
mean_img  = image.mean_img(image.concat_imgs(imp_imgs))
mask_img  = compute_brain_mask(mean_img, threshold=0.1)
mask_data = mask_img.get_fdata().astype(bool)
n_voxels  = mask_data.sum()
print(f'Brain mask: {n_voxels:,} voxels')

# ── Stack into (subjects × voxels) ────────────────────────────────────────────
X_imp = np.stack(
    [img.get_fdata()[mask_data] for img in imp_imgs],
    axis=0
)  # shape: (n_subjects, n_voxels)

print(f'Feature matrix: {X_imp.shape}  '
      f'(mean={X_imp.mean():.4f}, std={X_imp.std():.4f})')

In [ ]:
# ── Voxel-wise one-sample t-test against zero ─────────────────────────────────
t_stats, p_vals = stats.ttest_1samp(X_imp, popmean=0, axis=0)

print(f't-statistics: min={t_stats.min():.2f}  max={t_stats.max():.2f}')
print(f'Uncorrected p < 0.001: {(p_vals < 0.001).sum():,} voxels')

# Save unthresholded t-map
t_vol = np.zeros(mask_data.shape, dtype=np.float32)
t_vol[mask_data] = t_stats
t_map_img = nib.Nifti1Image(t_vol, ref_img.affine)
nib.save(t_map_img, OUT_DIR / 'group_tmap_unthresholded.nii.gz')
print('Saved: group_tmap_unthresholded.nii.gz')

In [ ]:
# ── FDR correction (Benjamini-Hochberg) ────────────────────────────────────────
reject_fdr, p_fdr, _, _ = multipletests(p_vals, alpha=ALPHA_FDR, method='fdr_bh')

n_sig = reject_fdr.sum()
print(f'FDR q < {ALPHA_FDR}: {n_sig:,} significant voxels '
      f'({n_sig / n_voxels * 100:.1f}% of brain)')

# Build thresholded map: keep t-value only where FDR-significant
t_fdr_flat = np.where(reject_fdr, t_stats, 0.0)
t_fdr_vol  = np.zeros(mask_data.shape, dtype=np.float32)
t_fdr_vol[mask_data] = t_fdr_flat
t_fdr_img  = nib.Nifti1Image(t_fdr_vol, ref_img.affine)
nib.save(t_fdr_img, OUT_DIR / 'group_tmap_fdr.nii.gz')
print('Saved: group_tmap_fdr.nii.gz')

---
## Section 3 — Visualise Group Map

Display the FDR-corrected t-map on a glass brain and an axial slice montage.
Any voxel shown survived `q < 0.05` (FDR) and had a positive mean weight
across subjects.

In [ ]:
# Glass brain
plotting.plot_glass_brain(
    t_map_img,
    colorbar=True,
    threshold=1,
    display_mode='lzry',
    cmap='hot',
    vmax=3,
    title=f'Group importance map — FDR q < {ALPHA_FDR}  '
          f'(n={len(imp_subs)} subjects)',
).savefig(OUT_DIR / 'glass_brain_fdr.png', dpi=150)
plt.show()

In [ ]:
# Axial slice montage
plotting.plot_stat_map(
    t_map_img,
    display_mode='z',
    cut_coords=8,
    colorbar=True,
    threshold=1,
    cmap='hot',
    vmax=3,
    title=f'FDR q < {ALPHA_FDR}  |  axial slices',
).savefig(OUT_DIR / 'axial_fdr.png', dpi=150)
plt.show()

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
print('=' * 55)
print('GROUP-LEVEL SUMMARY')
print('=' * 55)
print(f'  Subjects:               {len(acc_vals)}')
print(f'  Group accuracy:         {group_mean:.3f} ± {group_sd:.3f} SD')
print(f'  t-test vs chance:       '
      f't({len(acc_vals)-1}) = {t_stat:.2f},  p = {p_val:.4f}')
print(f'  Brain voxels tested:    {n_voxels:,}')
print(f'  Significant (FDR):      {n_sig:,} voxels')
print('\nOutput files:')
for f in sorted(OUT_DIR.iterdir()):
    print(f'  {f.name}')
print('=' * 55)